# RAG fallback notebook (Google Colab)

Use this if the laptop dies, the venv breaks, or Streamlit will not start.
It is the same pipeline as `app.py` with the Streamlit layer removed.

**Run order:** Setup -> Keys -> Upload -> Index -> Ask.

Open it at https://colab.research.google.com -> GitHub tab -> paste this repo URL.

## 1. Setup (~2 minutes)

In [ ]:
!pip install -q sentence-transformers faiss-cpu pypdf groq python-dotenv pandas \
    pytesseract pypdfium2 python-docx openpyxl
!apt-get install -qq tesseract-ocr > /dev/null
print('setup done')

## 2. Keys

`getpass` keeps the key out of the saved notebook, so it is safe to commit this file.

In [ ]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Groq API key (gsk_...): ')
os.environ['GEMINI_API_KEY'] = getpass('Gemini API key (optional, press Enter to skip): ')
print('keys set')

## 3. Pipeline

In [ ]:
import io, os, shutil
from dataclasses import dataclass

import faiss
import numpy as np
import pandas as pd
import requests
from groq import Groq
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

EMBED_MODEL = 'all-MiniLM-L6-v2'   # 'paraphrase-multilingual-MiniLM-L12-v2' if not English
GROQ_MODEL = 'openai/gpt-oss-120b'
GEMINI_MODEL = 'gemini-3.8-flash'
CHUNK_CHARS, CHUNK_OVERLAP, TOP_K = 1000, 150, 4
OCR_TRIGGER_CHARS = 40

SYSTEM_PROMPT = (
    'Answer the question using only the provided context. '
    'If the context does not contain the answer, say so plainly. '
    'Cite the source filename for each claim.'
)

embedder = SentenceTransformer(EMBED_MODEL)


@dataclass
class Chunk:
    text: str
    source: str


def split(text, source):
    chunks, start = [], 0
    text = ' '.join(text.split())
    while start < len(text):
        end = min(start + CHUNK_CHARS, len(text))
        if end < len(text):
            space = text.rfind(' ', start, end)
            if space > start:
                end = space
        piece = text[start:end].strip()
        if piece:
            chunks.append(Chunk(piece, source))
        if end >= len(text):
            break
        start = max(end - CHUNK_OVERLAP, start + 1)
    return chunks


def rows_to_chunks(df, source):
    return [
        Chunk('; '.join(f'{c}: {r[c]}' for c in df.columns if pd.notna(r[c])),
              f'{source} row {i + 1}')
        for i, r in df.iterrows()
    ]


def ocr_pdf(data):
    import pypdfium2 as pdfium, pytesseract
    return '\n'.join(
        pytesseract.image_to_string(p.render(scale=200 / 72).to_pil())
        for p in pdfium.PdfDocument(data)
    )


def load_document(name, data):
    low = name.lower()
    if low.endswith('.pdf'):
        reader = PdfReader(io.BytesIO(data))
        text = '\n'.join(p.extract_text() or '' for p in reader.pages)
        if len(text.strip()) < OCR_TRIGGER_CHARS * len(reader.pages):
            return split(ocr_pdf(data), name), 'OCR (scanned)'
        return split(text, name), 'text layer'
    if low.endswith('.csv'):
        return rows_to_chunks(pd.read_csv(io.BytesIO(data)), name), 'CSV rows'
    if low.endswith(('.xlsx', '.xlsm')):
        sheets = pd.read_excel(io.BytesIO(data), sheet_name=None)
        out = []
        for s, df in sheets.items():
            out += rows_to_chunks(df, f'{name}:{s}')
        return out, 'Excel'
    if low.endswith('.docx'):
        import docx
        d = docx.Document(io.BytesIO(data))
        parts = [p.text for p in d.paragraphs if p.text.strip()]
        for t in d.tables:
            for row in t.rows:
                parts.append(' | '.join(c.text.strip() for c in row.cells))
        return split('\n'.join(parts), name), 'Word'
    return split(data.decode('utf-8', errors='ignore'), name), 'plain text'


def build_index(chunks):
    v = np.asarray(embedder.encode([c.text for c in chunks],
                                   normalize_embeddings=True), dtype='float32')
    idx = faiss.IndexFlatIP(v.shape[1])
    idx.add(v)
    return idx


def retrieve(index, chunks, question, k=TOP_K):
    q = np.asarray(embedder.encode([question], normalize_embeddings=True), dtype='float32')
    _, ids = index.search(q, min(k, len(chunks)))
    return [chunks[i] for i in ids[0] if i >= 0]


def ask_groq(question, context):
    c = Groq(api_key=os.environ['GROQ_API_KEY'])
    r = c.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'system', 'content': SYSTEM_PROMPT},
                  {'role': 'user',
                   'content': f'Context:\n{context}\n\nQuestion: {question}'}],
        temperature=0.2)
    return r.choices[0].message.content


def ask_gemini(question, context):
    r = requests.post(
        f'https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent',
        headers={'x-goog-api-key': os.environ['GEMINI_API_KEY']},
        json={'systemInstruction': {'parts': [{'text': SYSTEM_PROMPT}]},
              'contents': [{'parts': [{'text': f'Context:\n{context}\n\nQuestion: {question}'}]}],
              'generationConfig': {'maxOutputTokens': 2048, 'temperature': 0.2}},
        timeout=90)
    r.raise_for_status()
    return r.json()['candidates'][0]['content']['parts'][0]['text']


def answer(question, context):
    if os.environ.get('GROQ_API_KEY'):
        try:
            return ask_groq(question, context), f'Groq / {GROQ_MODEL}'
        except Exception as exc:
            print(f'Groq unavailable ({type(exc).__name__}), falling back to Gemini.')
    if os.environ.get('GEMINI_API_KEY'):
        return ask_gemini(question, context), f'Gemini / {GEMINI_MODEL}'
    raise RuntimeError('No usable API key.')


print('pipeline ready')

## 4. Upload documents

In [ ]:
from google.colab import files

uploaded = files.upload()   # pick PDFs / CSV / DOCX / XLSX

chunks = []
for name, data in uploaded.items():
    got, how = load_document(name, data)
    chunks += got
    print(f'{name}: {how}, {len(got)} chunks')

index = build_index(chunks)
print(f'\nindexed {len(chunks)} chunks')

## 5. Ask

In [ ]:
question = 'How long do I have to claim a damaged item?'  # <- edit

hits = retrieve(index, chunks, question)
context = '\n\n'.join(f'[{c.source}] {c.text}' for c in hits)
text, model = answer(question, context)

print(text)
print(f'\n--- answered by {model} ---')
print('sources:', ', '.join(dict.fromkeys(c.source for c in hits)))

## 6. Optional: serve the real Streamlit app from Colab

Only if you need the UI. Clone the repo, add the key, and tunnel it out.

In [ ]:
# !git clone https://github.com/Geetarthjain15/HCL-Hackathon.git
# %cd HCL-Hackathon
# !pip install -q -r requirements.txt
# open('.env', 'w').write(f"GROQ_API_KEY={os.environ['GROQ_API_KEY']}\n")
# !npm install -g localtunnel
# !streamlit run app.py &>/content/log.txt &
# !curl -s ipv4.icanhazip.com   # this IP is the tunnel password
# !npx localtunnel --port 8501